### STEP 2. 실제 스키마·PK·FK를 먼저 검증

In [1]:
import pandas as pd

# 1. processed CSV 불러오기
customers = pd.read_csv("../data/processed/customers_clean.csv")
orders = pd.read_csv("../data/processed/orders_clean.csv")
order_items = pd.read_csv("../data/processed/order_items_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")


# 2. 각 테이블에 반드시 있어야 하는 컬럼 정의
required_columns = {
    "customers": [
        "customer_id", "name", "gender",
        "age", "city", "signup_date"
    ],
    "orders": [
        "order_id", "customer_id", "order_date",
        "payment_method", "order_status",
        "order_month", "order_dayofweek"
    ],
    "order_items": [
        "order_item_id", "order_id", "product_id",
        "quantity", "unit_price", "line_total"
    ],
    "products": [
        "product_id", "product_name", "category", "price"
    ],
}


# 3. 실제 데이터와 이름을 연결
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
}


# 4. 필수 컬럼이 모두 있는지 확인
for table_name, df in tables.items():
    missing_columns = set(required_columns[table_name]) - set(df.columns)

    if len(missing_columns) == 0:
        print(f"PASS | {table_name}: 필수 컬럼이 모두 존재합니다.")
    else:
        print(f"FAIL | {table_name}: 누락 컬럼 = {missing_columns}")

PASS | customers: 필수 컬럼이 모두 존재합니다.
PASS | orders: 필수 컬럼이 모두 존재합니다.
PASS | order_items: 필수 컬럼이 모두 존재합니다.
PASS | products: 필수 컬럼이 모두 존재합니다.


In [2]:
# STEP 2-2. PK 결측값 확인

# 각 테이블의 PK 컬럼
primary_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "products": "product_id",
}

# PK에 결측값이 있는지 확인
for table_name, pk in primary_keys.items():
    df = tables[table_name]

    null_count = df[pk].isna().sum()

    if null_count == 0:
        print(f"PASS | {table_name}.{pk}: PK 결측이 없습니다.")
    else:
        print(f"FAIL | {table_name}.{pk}: PK 결측 {null_count}개")

PASS | customers.customer_id: PK 결측이 없습니다.
PASS | orders.order_id: PK 결측이 없습니다.
PASS | order_items.order_item_id: PK 결측이 없습니다.
PASS | products.product_id: PK 결측이 없습니다.


In [3]:
# STEP 2-3. PK 중복 확인

for table_name, pk in primary_keys.items():
    df = tables[table_name]

    # PK에서 중복된 값의 개수 확인
    duplicate_count = df[pk].duplicated().sum()

    if duplicate_count == 0:
        print(f"PASS | {table_name}.{pk}: PK 중복이 없습니다.")
    else:
        print(f"FAIL | {table_name}.{pk}: PK 중복 {duplicate_count}개")

PASS | customers.customer_id: PK 중복이 없습니다.
PASS | orders.order_id: PK 중복이 없습니다.
PASS | order_items.order_item_id: PK 중복이 없습니다.
PASS | products.product_id: PK 중복이 없습니다.


In [4]:
# STEP 2-4. 부모 key 고유성 확인

parent_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
}

for table_name, key in parent_keys.items():
    df = tables[table_name]

    # 부모 key의 중복 개수 확인
    duplicate_count = df[key].duplicated().sum()

    if duplicate_count == 0:
        print(f"PASS | {table_name}.{key}: 부모 key가 고유합니다.")
    else:
        print(f"FAIL | {table_name}.{key}: 중복 {duplicate_count}개")

PASS | customers.customer_id: 부모 key가 고유합니다.
PASS | orders.order_id: 부모 key가 고유합니다.
PASS | products.product_id: 부모 key가 고유합니다.


In [5]:
# STEP 2-5. FK 결측값 확인

foreign_keys = {
    "orders": ["customer_id"],
    "order_items": ["order_id", "product_id"],
}

for table_name, fk_list in foreign_keys.items():
    df = tables[table_name]

    for fk in fk_list:
        null_count = df[fk].isna().sum()

        if null_count == 0:
            print(f"PASS | {table_name}.{fk}: FK 결측이 없습니다.")
        else:
            print(f"FAIL | {table_name}.{fk}: FK 결측 {null_count}개")

PASS | orders.customer_id: FK 결측이 없습니다.
PASS | order_items.order_id: FK 결측이 없습니다.
PASS | order_items.product_id: FK 결측이 없습니다.


In [6]:
# STEP 2-6. 부모 테이블에 없는 FK 확인

fk_checks = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
]

for child_table, fk, parent_table, parent_key in fk_checks:
    child_df = tables[child_table]
    parent_df = tables[parent_table]

    # 부모 테이블에 존재하지 않는 FK 찾기
    invalid_fk = ~child_df[fk].isin(parent_df[parent_key])
    invalid_count = invalid_fk.sum()

    if invalid_count == 0:
        print(f"PASS | {child_table}.{fk}: 부모 테이블에 없는 FK가 없습니다.")
    else:
        print(f"FAIL | {child_table}.{fk}: 부모 테이블에 없는 FK {invalid_count}개")

FAIL | orders.customer_id: 부모 테이블에 없는 FK 2개
FAIL | order_items.order_id: 부모 테이블에 없는 FK 2개
FAIL | order_items.product_id: 부모 테이블에 없는 FK 26개


In [7]:
# 부모 테이블에 없는 FK 값 확인

for child_table, fk, parent_table, parent_key in fk_checks:
    child_df = tables[child_table]
    parent_df = tables[parent_table]

    # 부모 테이블에 존재하지 않는 행만 찾기
    invalid_rows = child_df[
        ~child_df[fk].isin(parent_df[parent_key])
    ]

    print(f"\n[{child_table}.{fk}]")
    print("부모 테이블에 없는 FK 값:")
    print(invalid_rows[fk].unique())


[orders.customer_id]
부모 테이블에 없는 FK 값:
[9991 9992]

[order_items.order_id]
부모 테이블에 없는 FK 값:
[9991 9992]

[order_items.product_id]
부모 테이블에 없는 FK 값:
[  73   48   18   95 9991 9992]


"FK 검증 결과 참조 무결성 문제가 발견되었다."

In [8]:
# STEP 2-7. 부모 테이블에 없는 FK를 가진 행 확인

for child_table, fk, parent_table, parent_key in fk_checks:
    child_df = tables[child_table]
    parent_df = tables[parent_table]

    # 부모 테이블에 없는 FK를 가진 행 찾기
    invalid_rows = child_df[
        ~child_df[fk].isin(parent_df[parent_key])
    ]

    print(f"\n[{child_table}.{fk}]")
    print(f"문제 행 수: {len(invalid_rows)}")

    display(invalid_rows)


[orders.customer_id]
문제 행 수: 2


,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
125,126,9991,2026-01-13,naver_pay,completed,2026-01,Tuesday
277,278,9992,2026-07-10,kakao_pay,cancelled,2026-07,Friday



[order_items.order_id]
문제 행 수: 2


,order_item_id,order_id,product_id,quantity,unit_price,line_total
147,150,9991,89,5.0,178000.0,890000.0
609,620,9992,98,1.0,32000.0,32000.0



[order_items.product_id]
문제 행 수: 26


,order_item_id,order_id,product_id,quantity,unit_price,line_total
20,22,9,73,1.0,156000.0,156000.0
30,32,14,48,1.0,25000.0,25000.0
75,77,36,18,1.0,140000.0,140000.0
88,90,41,48,5.0,25000.0,125000.0
96,99,44,48,4.0,25000.0,100000.0
103,106,47,48,3.0,25000.0,75000.0
127,130,57,48,3.0,25000.0,75000.0
210,214,86,95,1.0,82000.0,82000.0
238,242,97,95,3.0,82000.0,246000.0
256,260,105,9991,1.0,76000.0,76000.0


In [9]:
# 부모 products에 없는 product_id를 고유값별로 확인

invalid_product_rows = order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

# 문제 product_id별 등장 횟수
invalid_product_rows["product_id"].value_counts()

product_id
48      9
73      6
18      5
95      4
9991    1
9992    1
Name: count, dtype: int64

In [10]:
# 문제가 된 product_id가 products에 실제로 존재하는지 확인

problem_ids = [18, 48, 73, 95, 9991, 9992]

products[
    products["product_id"].isin(problem_ids)
]

,product_id,product_name,category,price


order_items.product_id 검증 결과 26개 행에서 부모 products에 존재하지 않는 FK가 발견되어 참조 무결성 검증에 실패했다. 중복을 제외하면 문제 product_id는 18, 48, 73, 95, 9991, 9992이다.

### STEP 3. merge와 completed 집계의 Business Rule 검증

In [11]:
# STEP 3-1. line_total 계산이 맞는지 확인

# 우리가 직접 계산한 금액
calculated_total = (
    order_items["quantity"] * order_items["unit_price"]
)

# 저장된 line_total과 계산한 금액이 다른 행 찾기
wrong_rows = order_items[
    order_items["line_total"] != calculated_total
]

print(f"전체 행 수: {len(order_items)}")
print(f"금액 불일치 행 수: {len(wrong_rows)}")

if len(wrong_rows) == 0:
    print("PASS | line_total = quantity × unit_price")
else:
    print("FAIL | line_total 계산이 맞지 않는 행이 있습니다.")
    display(wrong_rows)

전체 행 수: 752
금액 불일치 행 수: 0
PASS | line_total = quantity × unit_price


In [12]:
# STEP 3-2. merge 기준 key 고유성 확인

keys_to_check = {
    "orders": (orders, "order_id"),
    "products": (products, "product_id"),
}

for table_name, (df, key) in keys_to_check.items():
    duplicate_count = df[key].duplicated().sum()

    if duplicate_count == 0:
        print(f"PASS | {table_name}.{key}: 고유합니다.")
    else:
        print(f"FAIL | {table_name}.{key}: 중복 {duplicate_count}개")

PASS | orders.order_id: 고유합니다.
PASS | products.product_id: 고유합니다.


In [13]:
# STEP 3-3. order_items와 orders merge 검증

# merge 전 행 수
before_rows = len(order_items)

# 주문 정보를 order_items에 연결
merged_orders = order_items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one"  # 여러 주문상세 → 하나의 주문
)

# merge 후 행 수
after_rows = len(merged_orders)

print(f"merge 전 행 수: {before_rows}")
print(f"merge 후 행 수: {after_rows}")

if before_rows == after_rows:
    print("PASS | merge 전후 행 수가 같습니다.")
else:
    print("FAIL | merge 후 행 수가 달라졌습니다.")

merge 전 행 수: 752
merge 후 행 수: 752
PASS | merge 전후 행 수가 같습니다.


In [14]:
# STEP 3-4. orders merge 미매칭 확인

# orders에서 가져온 order_status가 비어 있으면
# 해당 order_id가 orders와 매칭되지 않았다는 뜻
unmatched_rows = merged_orders[
    merged_orders["order_status"].isna()
]

print(f"미매칭 행 수: {len(unmatched_rows)}")

if len(unmatched_rows) == 0:
    print("PASS | 모든 order_id가 orders와 매칭되었습니다.")
else:
    print("FAIL | orders와 매칭되지 않은 행이 있습니다.")
    display(unmatched_rows)

미매칭 행 수: 2
FAIL | orders와 매칭되지 않은 행이 있습니다.


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
147,150,9991,89,5.0,178000.0,890000.0,NaN,NaN,NaN,NaN,NaN,NaN
609,620,9992,98,1.0,32000.0,32000.0,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# STEP 3-5. products merge 및 category 결측 확인

merged_products = order_items.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# merge 전후 행 수 확인
print(f"merge 전 행 수: {len(order_items)}")
print(f"merge 후 행 수: {len(merged_products)}")

# category가 없는 행 확인
missing_category = merged_products[
    merged_products["category"].isna()
]

print(f"category 결측 행 수: {len(missing_category)}")

if len(missing_category) == 0:
    print("PASS | 모든 상품의 category가 존재합니다.")
else:
    print("FAIL | category가 없는 상품이 있습니다.")
    display(missing_category)

merge 전 행 수: 752
merge 후 행 수: 752
category 결측 행 수: 26
FAIL | category가 없는 상품이 있습니다.


,order_item_id,order_id,product_id,quantity,unit_price,line_total,category
20,22,9,73,1.0,156000.0,156000.0,NaN
30,32,14,48,1.0,25000.0,25000.0,NaN
75,77,36,18,1.0,140000.0,140000.0,NaN
88,90,41,48,5.0,25000.0,125000.0,NaN
96,99,44,48,4.0,25000.0,100000.0,NaN
103,106,47,48,3.0,25000.0,75000.0,NaN
127,130,57,48,3.0,25000.0,75000.0,NaN
210,214,86,95,1.0,82000.0,82000.0,NaN
238,242,97,95,3.0,82000.0,246000.0,NaN
256,260,105,9991,1.0,76000.0,76000.0,NaN


In [16]:
# STEP 3-6. order_date 날짜 변환 실패 확인

# 날짜로 변환
# 변환할 수 없는 값은 NaT로 처리
order_date_parsed = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

# 원래 값은 있는데 날짜 변환에 실패한 행 찾기
date_failed = orders[
    orders["order_date"].notna() & order_date_parsed.isna()
]

print(f"날짜 변환 실패 행 수: {len(date_failed)}")

if len(date_failed) == 0:
    print("PASS | 모든 order_date가 날짜로 변환되었습니다.")
else:
    print("FAIL | 날짜로 변환할 수 없는 값이 있습니다.")
    display(date_failed)

날짜 변환 실패 행 수: 0
PASS | 모든 order_date가 날짜로 변환되었습니다.


In [17]:
# STEP 3-7. completed 주문 범위 확인

# 주문 상태별 건수 확인
print("주문 상태별 건수")
display(orders["order_status"].value_counts(dropna=False))

# completed 주문만 추출
completed_orders = orders[
    orders["order_status"] == "completed"
].copy()

print(f"전체 주문 수: {len(orders)}")
print(f"completed 주문 수: {len(completed_orders)}")

if len(completed_orders) > 0:
    print("PASS | completed 주문이 존재합니다.")
else:
    print("FAIL | completed 주문이 없습니다.")

주문 상태별 건수


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

전체 주문 수: 300
completed 주문 수: 180
PASS | completed 주문이 존재합니다.


In [18]:
# STEP 3-8. completed 주문의 source total 계산

# order_items에 주문 상태 연결
completed_items = order_items.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

# completed 주문의 상세 행만 선택
completed_items = completed_items[
    completed_items["order_status"] == "completed"
].copy()

# completed 주문 상세 금액의 전체 합계
source_total = completed_items["line_total"].sum()

print(f"completed 상세 행 수: {len(completed_items)}")
print(f"completed source total: {source_total:,.0f}원")

completed 상세 행 수: 446
completed source total: 141,340,000원


In [19]:
# STEP 3-9. completed 주문의 카테고리별 금액 집계

category_items = completed_items.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# category가 없는 행 수 확인
missing_category_count = category_items["category"].isna().sum()

print(f"category 결측 행 수: {missing_category_count}")

# 카테고리별 금액 합계
category_sales = (
    category_items
    .groupby("category", as_index=False)["line_total"]
    .sum()
)

display(category_sales)

# 카테고리별 집계 금액의 전체 합
category_grouped_total = category_sales["line_total"].sum()

print(f"completed source total   : {source_total:,.0f}원")
print(f"category grouped total   : {category_grouped_total:,.0f}원")

if source_total == category_grouped_total:
    print("PASS | 카테고리별 합계가 source total과 같습니다.")
else:
    print("FAIL | 카테고리별 합계가 source total과 다릅니다.")

category 결측 행 수: 16


,category,line_total
0,도서,17663000.0
1,뷰 티,648000.0
2,뷰티,17103000.0
3,생활 용품,2108000.0
4,생활용품,7001000.0
5,스포츠,13237000.0
6,식품,25035000.0
7,전자 기기,105000.0
8,전자기기,21485000.0
9,패션,32940000.0


completed source total   : 141,340,000원
category grouped total   : 137,325,000원
FAIL | 카테고리별 합계가 source total과 다릅니다.


In [22]:
# STEP 3-10. completed 주문의 월별 금액 집계

# order_items에 주문 상태와 주문 날짜 연결
monthly_items = order_items.merge(
    orders[["order_id", "order_status", "order_date"]],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

# completed 주문만 선택
monthly_items = monthly_items[
    monthly_items["order_status"] == "completed"
].copy()

# 날짜 형식으로 변환
monthly_items["order_date"] = pd.to_datetime(
    monthly_items["order_date"],
    errors="coerce"
)

# 연-월 만들기
monthly_items["order_month"] = (
    monthly_items["order_date"]
    .dt.to_period("M")
    .astype(str)
)

# 월별 금액 합계
monthly_sales = (
    monthly_items
    .groupby("order_month", as_index=False)["line_total"]
    .sum()
)

display(monthly_sales)

# 월별 집계 전체 합
monthly_grouped_total = monthly_sales["line_total"].sum()

print(f"completed source total : {source_total:,.0f}원")
print(f"monthly grouped total  : {monthly_grouped_total:,.0f}원")

if source_total == monthly_grouped_total:
    print("PASS | 월별 합계가 source total과 같습니다.")
else:
    print("FAIL | 월별 합계가 source total과 다릅니다.")

,order_month,line_total
0,2025-07,9561000.0
1,2025-08,9362000.0
2,2025-09,5299000.0
3,2025-10,12629000.0
4,2025-11,9583000.0
5,2025-12,8581000.0
6,2026-01,8135000.0
7,2026-02,7689000.0
8,2026-03,7247000.0
9,2026-04,11459000.0


completed source total : 141,340,000원
monthly grouped total  : 139,437,000원
FAIL | 월별 합계가 source total과 다릅니다.


In [23]:
# 월별 집계에서 빠진 completed 행 확인

missing_date_rows = monthly_items[
    monthly_items["order_date"].isna()
]

print(f"날짜가 없는 completed 상세 행 수: {len(missing_date_rows)}")
print(f"빠진 금액 합계: {missing_date_rows['line_total'].sum():,.0f}원")

display(missing_date_rows)

날짜가 없는 completed 상세 행 수: 6
빠진 금액 합계: 1,903,000원


,order_item_id,order_id,product_id,quantity,unit_price,line_total,order_status,order_date,order_month
163,167,72,92,3.0,74000.0,222000.0,completed,NaT,NaN
164,168,72,74,5.0,129000.0,645000.0,completed,NaT,NaN
532,542,214,47,1.0,188000.0,188000.0,completed,NaT,NaN
533,543,214,91,4.0,125000.0,500000.0,completed,NaT,NaN
534,544,214,83,3.0,74000.0,222000.0,completed,NaT,NaN
535,545,214,84,3.0,42000.0,126000.0,completed,NaT,NaN


검증 결과: completed source total과 monthly grouped total이 1,903,000원 차이 났다. 확인 결과 completed 주문 중 order_date가 결측인 상세 행의 금액 합계가 정확히 1,903,000원이었다. 따라서 날짜가 없는 데이터가 월별 groupby에서 제외된 것이 불일치 원인이다.

In [24]:
# STEP 3-11. Business Rule 최종 검증

print("[completed 매출 집계 검증]")
print(f"source total           : {source_total:,.0f}원")
print(f"category grouped total : {category_grouped_total:,.0f}원")
print(f"monthly grouped total  : {monthly_grouped_total:,.0f}원")

# 세 금액이 모두 같은지 확인
if source_total == category_grouped_total == monthly_grouped_total:
    print("\nPASS | 세 집계 금액이 모두 같습니다.")
else:
    print("\nFAIL | 세 집계 금액이 일치하지 않습니다.")

    print(
        f"카테고리 차이: "
        f"{source_total - category_grouped_total:,.0f}원"
    )

    print(
        f"월별 차이: "
        f"{source_total - monthly_grouped_total:,.0f}원"
    )

[completed 매출 집계 검증]
source total           : 141,340,000원
category grouped total : 137,325,000원
monthly grouped total  : 139,437,000원

FAIL | 세 집계 금액이 일치하지 않습니다.
카테고리 차이: 4,015,000원
월별 차이: 1,903,000원


In [25]:
# STEP 3-12. 카테고리 집계에서 빠진 금액 확인

missing_category_rows = category_items[
    category_items["category"].isna()
]

missing_category_total = missing_category_rows["line_total"].sum()

print(f"category가 없는 completed 상세 행 수: {len(missing_category_rows)}")
print(f"category가 없어 빠진 금액: {missing_category_total:,.0f}원")

print(
    f"source와 category 집계의 차이: "
    f"{source_total - category_grouped_total:,.0f}원"
)

# 두 금액이 같은지 확인
if missing_category_total == source_total - category_grouped_total:
    print("PASS | 카테고리 집계 차이의 원인을 확인했습니다.")
else:
    print("FAIL | 다른 원인이 추가로 존재합니다.")

display(missing_category_rows)

category가 없는 completed 상세 행 수: 16
category가 없어 빠진 금액: 4,015,000원
source와 category 집계의 차이: 4,015,000원
PASS | 카테고리 집계 차이의 원인을 확인했습니다.


,order_item_id,order_id,product_id,quantity,unit_price,line_total,order_status,category
16,22,9,73,1.0,156000.0,156000.0,completed,NaN
54,77,36,18,1.0,140000.0,140000.0,completed,NaN
77,106,47,48,3.0,25000.0,75000.0,completed,NaN
134,214,86,95,1.0,82000.0,82000.0,completed,NaN
164,260,105,9991,1.0,76000.0,76000.0,completed,NaN
193,323,130,48,2.0,25000.0,50000.0,completed,NaN
237,412,164,73,3.0,156000.0,468000.0,completed,NaN
258,454,179,18,3.0,140000.0,420000.0,completed,NaN
285,496,196,73,2.0,156000.0,312000.0,completed,NaN
289,504,199,73,5.0,156000.0,780000.0,completed,NaN


### STEP 3 검증 결과

- completed source total: 141,340,000원
- category grouped total: 137,325,000원
- monthly grouped total: 139,437,000원

최종 판정: FAIL

#### 원인
- 카테고리 집계 차이: 4,015,000원
  - category가 없는 completed 상세 행이 groupby에서 제외됨
  - 빠진 금액과 집계 차액이 일치함

- 월별 집계 차이: 1,903,000원
  - order_date가 없는 completed 상세 행이 groupby에서 제외됨
  - 빠진 금액과 집계 차액이 일치함

따라서 집계 계산 자체의 오류가 아니라,
원본 데이터의 FK/결측 문제로 인해 집계 결과가 일치하지 않았다.

또한 현재 계산한 금액은 completed 주문의 상세 금액 합계이며,
할인·배송비·세금·부분 환불·정산 정보가 없으므로
회계상 순매출로 해석하지 않는다.

### STEP 4. Generated Code를 실행하지 않고 AST Static Scan

In [29]:
import ast
import pandas as pd



class ASTStaticScanner(ast.NodeVisitor):

    def __init__(self):
        self.findings = []

    def visit_Call(self, node):
        # 1. 동적 코드 실행 탐지 (eval, exec, compile)
        if (
            isinstance(node.func, ast.Name)
            and node.func.id in ["eval", "exec", "compile"]
        ):
            self.findings.append(
                {
                    "Category": "Dynamic Execution",
                    "Target": node.func.id,
                    "Severity": "CRITICAL",
                    "Line": node.lineno,
                }
            )

        # 2. OS 및 Shell 명령 실행 (os.system, subprocess 등)
        elif isinstance(node.func, ast.Attribute):
            if isinstance(
                node.func.value, ast.Name
            ) and node.func.value.id in ["os", "subprocess"]:
                self.findings.append(
                    {
                        "Category": "OS/Shell Command",
                        "Target": f"{node.func.value.id}.{node.func.attr}",
                        "Severity": "CRITICAL",
                        "Line": node.lineno,
                    }
                )

            # 3. 파일 쓰기/삭제/교체 (to_csv, to_excel, write_text 등)
            elif node.func.attr in [
                "to_csv",
                "to_excel",
                "to_parquet",
                "write_text",
                "write_bytes",
                "remove",
                "unlink",
                "rmdir",
            ]:
                self.findings.append(
                    {
                        "Category": "File Write/Delete",
                        "Target": node.func.attr,
                        "Severity": "HIGH",
                        "Line": node.lineno,
                    }
                )

            # 4. 외부 네트워크 요청 (requests, urllib, socket 등)
            elif isinstance(
                node.func.value, ast.Name
            ) and node.func.value.id in ["requests", "urllib", "socket", "http"]:
                self.findings.append(
                    {
                        "Category": "Network Request",
                        "Target": f"{node.func.value.id}.{node.func.attr}",
                        "Severity": "HIGH",
                        "Line": node.lineno,
                    }
                )

        self.generic_visit(node)

    def visit_Import(self, node):
        # 5. 위험 패키지 import 탐지 (pip, ensurepip, subprocess 등)
        for alias in node.names:
            if alias.name in ["pip", "ensurepip", "subprocess", "socket"]:
                self.findings.append(
                    {
                        "Category": "Package/System Import",
                        "Target": alias.name,
                        "Severity": "HIGH",
                        "Line": node.lineno,
                    }
                )
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module in ["pip", "ensurepip", "subprocess", "socket"]:
            self.findings.append(
                {
                    "Category": "Package/System Import",
                    "Target": node.module,
                    "Severity": "HIGH",
                    "Line": node.lineno,
                }
            )
        self.generic_visit(node)

    def visit_Constant(self, node):
        # 6. 하드코딩된 API Key / Token / Password 탐지
        if isinstance(node.value, str):
            val_lower = node.value.lower()
            if any(
                keyword in val_lower
                for keyword in [
                    "api_key",
                    "bearer ",
                    "sk-",
                    "password=",
                    "secret_key",
                ]
            ):
                self.findings.append(
                    {
                        "Category": "Hardcoded Credential",
                        "Target": node.value[:15] + "...",
                        "Severity": "CRITICAL",
                        "Line": node.lineno,
                    }
                )
        self.generic_visit(node)



def scan_generated_code(code_string: str) -> pd.DataFrame:
    """코드를 실행하지 않고 구문(AST)을 파싱하여 위험 요소를 정적 검사합니다."""
    try:
        tree = ast.parse(code_string)
        scanner = ASTStaticScanner()
        scanner.visit(tree)

        if not scanner.findings:
            return pd.DataFrame(
                [
                    {
                        "Category": "None",
                        "Target": "No risks detected",
                        "Severity": "REVIEW",
                        "Line": "-",
                    }
                ]
            )
        return pd.DataFrame(scanner.findings)
    except SyntaxError as e:
        return pd.DataFrame(
            [
                {
                    "Category": "Syntax Error",
                    "Target": str(e),
                    "Severity": "BLOCKED",
                    "Line": e.lineno,
                }
            ]
        )



# ==========================================
# 테스트용 정적 스캔 예제 실행
# ==========================================
DEFAULT_STATIC_SCAN_EXAMPLE = """
import os
import requests
import pip

api_token = "sk-1234567890abcdef"

eval("print('danger')")
os.system("rm -rf /")
response = requests.get("https://example.com/api")
df.to_csv("data/output.csv")
"""

print("=== [STEP 4] AST Static Scan 결과 ===")
scan_result = scan_generated_code(DEFAULT_STATIC_SCAN_EXAMPLE)
display(scan_result)

=== [STEP 4] AST Static Scan 결과 ===


,Category,Target,Severity,Line
0,Package/System Import,pip,HIGH,4
1,Hardcoded Credential,sk-1234567890ab...,CRITICAL,6
2,Dynamic Execution,eval,CRITICAL,8
3,OS/Shell Command,os.system,CRITICAL,9
4,Network Request,requests.get,HIGH,10
5,File Write/Delete,to_csv,HIGH,11


In [30]:
# STEP 5-1. 회귀 Feature Contract 정의

# 실제 모델에 사용하려는 Feature
regression_features = [
    "payment_method",
    "order_month",
    "order_dayofweek",
    "gender",
    "age",
    "city",
]

# 회귀 문제에서 사용하면 안 되는 Feature
regression_forbidden = [
    "order_total",
    "line_total",
    "quantity",
    "unit_price",
    "item_count",
    "total_quantity",
    "avg_unit_price",
    "order_status",
    "order_id",
    "customer_id",
]

print("사용 예정 Feature:")
print(regression_features)

print("\n금지 Feature:")
print(regression_forbidden)

사용 예정 Feature:
['payment_method', 'order_month', 'order_dayofweek', 'gender', 'age', 'city']

금지 Feature:
['order_total', 'line_total', 'quantity', 'unit_price', 'item_count', 'total_quantity', 'avg_unit_price', 'order_status', 'order_id', 'customer_id']


In [31]:
# STEP 5-2. 회귀 Feature Contract 검증

# 사용 예정 Feature 중 금지 Feature 찾기
forbidden_used = [
    feature
    for feature in regression_features
    if feature in regression_forbidden
]

if len(forbidden_used) == 0:
    print("PASS | 회귀 모델에 금지 Feature가 없습니다.")
else:
    print("FAIL | 회귀 모델에 금지 Feature가 포함되어 있습니다.")
    print("금지 Feature:", forbidden_used)

PASS | 회귀 모델에 금지 Feature가 없습니다.


In [32]:
# STEP 5-3. 회귀 Feature Audit 표 만들기

regression_audit = pd.DataFrame({
    "feature": regression_features,
    "prediction_time_available": [
        "YES",  # payment_method
        "YES",  # order_month
        "YES",  # order_dayofweek
        "YES",  # gender
        "YES",  # age
        "YES",  # city
    ],
    "forbidden": [
        "NO",
        "NO",
        "NO",
        "NO",
        "NO",
        "NO",
    ],
    "decision": [
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
        "USE",
    ],
})

display(regression_audit)

,feature,prediction_time_available,forbidden,decision
0,payment_method,YES,NO,USE
1,order_month,YES,NO,USE
2,order_dayofweek,YES,NO,USE
3,gender,YES,NO,USE
4,age,YES,NO,USE
5,city,YES,NO,USE


In [33]:
# STEP 5-4. 분류 Feature Contract 정의

# 분류 모델에 사용하려는 Feature
classification_features = [
    "payment_method",
    "item_count",
    "total_quantity",
    "order_amount",
    "age",
    "city",
]

# 분류 문제에서 사용하면 안 되는 Feature
classification_forbidden = [
    "order_status",
    "is_cancelled",
    "order_id",
    "customer_id",
    "product_id",
    "line_total",
    "quantity",
    "unit_price",
    "cancel_reason",
    "cancelled_at",
]

# 예측 시점 가정에 따라 검토가 필요한 Feature
classification_review = [
    "item_count",
    "total_quantity",
    "order_amount",
]

print("사용 예정 Feature:")
print(classification_features)

print("\n금지 Feature:")
print(classification_forbidden)

print("\nREVIEW 대상:")
print(classification_review)

사용 예정 Feature:
['payment_method', 'item_count', 'total_quantity', 'order_amount', 'age', 'city']

금지 Feature:
['order_status', 'is_cancelled', 'order_id', 'customer_id', 'product_id', 'line_total', 'quantity', 'unit_price', 'cancel_reason', 'cancelled_at']

REVIEW 대상:
['item_count', 'total_quantity', 'order_amount']


In [34]:
# STEP 5-5. 분류 Feature Contract 검증

classification_result = []

for feature in classification_features:
    
    if feature in classification_forbidden:
        decision = "BLOCK"
        
    elif feature in classification_review:
        decision = "REVIEW"
        
    else:
        decision = "USE"
    
    classification_result.append({
        "feature": feature,
        "decision": decision
    })

classification_audit = pd.DataFrame(classification_result)

display(classification_audit)

,feature,decision
0,payment_method,USE
1,item_count,REVIEW
2,total_quantity,REVIEW
3,order_amount,REVIEW
4,age,USE
5,city,USE


In [35]:
# STEP 5-6. 회귀와 분류 Feature Contract 비교

feature_contract_comparison = pd.DataFrame({
    "feature": [
        "order_status",
        "order_id",
        "customer_id",
        "item_count",
        "total_quantity",
        "order_amount",
        "payment_method",
        "age",
        "city",
    ],
    "regression": [
        "BLOCK",
        "BLOCK",
        "BLOCK",
        "BLOCK",
        "BLOCK",
        "BLOCK",
        "USE",
        "USE",
        "USE",
    ],
    "classification": [
        "BLOCK",
        "BLOCK",
        "BLOCK",
        "REVIEW",
        "REVIEW",
        "REVIEW",
        "USE",
        "USE",
        "USE",
    ],
})

display(feature_contract_comparison)

,feature,regression,classification
0,order_status,BLOCK,BLOCK
1,order_id,BLOCK,BLOCK
2,customer_id,BLOCK,BLOCK
3,item_count,BLOCK,REVIEW
4,total_quantity,BLOCK,REVIEW
5,order_amount,BLOCK,REVIEW
6,payment_method,USE,USE
7,age,USE,USE
8,city,USE,USE


Feature Leakage는 단순히 컬럼 이름만 보고 판단하는 문제가 아니라 Prediction Time의 문제이다. 같은 Feature라도 무엇을 Target으로 예측하는지, 그리고 예측 시점에 실제로 알 수 있는 정보인지에 따라 사용 가능 여부가 달라질 수 있다. 예를 들어 order_amount는 주문 금액을 예측하는 회귀 문제에서는 정답과 직접적으로 연결되므로 사용할 수 없지만, 주문 취소 여부를 예측하는 분류 문제에서는 주문 생성 시점에 이미 금액이 확정되어 있다는 가정이라면 사용할 가능성이 있다. 따라서 Feature를 검토할 때는 컬럼 이름뿐 아니라 Target과 Prediction Time을 함께 확인해야 한다.

STEP 6. Sandbox와 Package 검토